# Verdicts: gate what gets weighted, record the negatives

Issue #135, the last child of the "what is actually predictive" epic (#114). Three signals — `defense_vs_position` (#132), `game_environment` (#133), `player_role_trend` (#134) — each ran through `weekly_backtest.score_signal` (#131) independently and each landed a verdict in its own module docstring. None of the three was checked against the same numeric bar, and none was checked against the actual bar the epic sets: beating the vendor's own weekly projection, not a proxy for it. This notebook does both — states one promotion rule before looking at any of the three signals' numbers, then applies it identically to all three.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from src.query import q
from src.gold.weekly_backtest import score_signal
from src.gold.league_scoring import league_points, STAT_COLUMNS

pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 100)

## The promotion rule, stated before any of the three signals' numbers below are computed

A signal-position cell promotes into `weekly_player_context` as a scored input if, scored by `weekly_backtest.score_signal` against the **`sleeper_points` baseline** — holding the vendor's own weekly projection fixed, which is the literal question this epic exists to answer, not a proxy for it — its incremental correlation clears **`|t_stat| >= 2.0`**, the identical reliability threshold `player_archetypes.py`'s `is_reliable` gate already uses for the same kind of decision. That alone isn't enough on a handful of clustered weeks, the same reason `player_archetypes.py` pairs its own `|t| >= 2` with a `_MIN_CELL_ROWS` floor: `score_signal` clusters its significance test by `(season, week)`, so the quantity that has to be adequately large here is `n_weeks`, not raw row count. The floor is **`n_weeks >= 17`** — one full season's worth of independently clustered weeks (the current NFL regular season), on the reasoning that a t-test drawn from less evidence than a single season is exactly the kind of small-sample significance `draft_strategy.py`'s slope check and `player_archetypes.py`'s own gate already exist to refuse trusting.

No exceptions, and the threshold is not re-derived per signal: every cell below, for all three signals, is checked against these same two constants.

In [2]:
GATE_MIN_ABS_T = 2.0
GATE_MIN_WEEKS = 17


def passes_gate(row) -> bool:
    return bool(row["n_weeks"] >= GATE_MIN_WEEKS and abs(row["t_stat"]) >= GATE_MIN_ABS_T)

## Building the inputs every signal is scored against

`actuals` (real fantasy points, both leagues' own scoring) and `projection` (`weekly_projections.sleeper_points`, the vendor baseline) are the same two frames all three of #132/#133/#134 already build, reused here rather than re-derived — the identical `build_actuals`/`build_projection` pattern `notebooks/defense_matchup.ipynb` established.

In [3]:
LEAGUES = q("SELECT * FROM league_settings")
LEAGUE_SCORING = {"sleeper": "half_ppr", "espn": "ppr"}

STATS = q(f"""
    SELECT player_id, season, week, position, {", ".join(STAT_COLUMNS)}
    FROM weekly_stats
    WHERE season_type = 'REG' AND position IN ('QB', 'RB', 'WR', 'TE')
""")


def build_actuals(league_row):
    out = STATS[["player_id", "season", "week", "position"]].copy()
    out["actual_points"] = league_points(STATS, league_row)
    return out


def build_projection(league_key, population):
    real = q(
        "SELECT player_id, season, week, sleeper_points FROM weekly_projections WHERE scoring = ?",
        [LEAGUE_SCORING[league_key]],
    )
    return population[["player_id", "season", "week"]].merge(
        real, on=["player_id", "season", "week"], how="left"
    )

## Same footing: all three signals against the two baselines that can actually be measured today

The epic's own bar (`sleeper_points`) is checked further down and, as the next section shows, can't complete on any of the three yet. Before that, this puts every signal's headline column — the one each of #132/#133/#134 already settled on as its own finding — on one comparison table, against the two walk-forward proxy baselines (`season_to_date_ppg`, `last3_ppg`) that eleven seasons of history *can* measure. Every number below is recomputed here, not copied from a docstring.

- `defense_vs_position` — `points_allowed_per_game_season_to_date`, the window #132 found matches or   leads the narrower recency windows at every position.
- `game_environment` — `implied_margin` (the confirmed RB/TE gamescript effect) and `wind` (the   confirmed QB/WR effect), plus `implied_team_total` and `temp`, the two #133 found baseline-fragile.
- `player_role_trend` — `target_share_delta` and `snap_share_delta`, representative of the seven   metrics #134 checked; every one of the seven shows the same season-to-date/last-3 sign flip,   exhaustively checked there across all seven metrics and windows 1-6, not re-derived here.

In [4]:
def build_dvp_signal(league_key):
    return q("""
        SELECT ws.player_id, ws.season, ws.week,
               dvp.points_allowed_per_game_season_to_date AS signal_value
        FROM weekly_stats ws
        JOIN defense_vs_position dvp
          ON dvp.league_key = ?
         AND dvp.season = ws.season AND dvp.week = ws.week
         AND dvp.defense_team = ws.opponent_team AND dvp.position = ws.position
        WHERE ws.season_type = 'REG' AND ws.position IN ('QB', 'RB', 'WR', 'TE')
    """, [league_key])


def build_ge_signal(column):
    return q(f"""
        SELECT ws.player_id, ws.season, ws.week, ge.{column} AS signal_value
        FROM weekly_stats ws
        JOIN game_environment ge
          ON ge.season = ws.season AND ge.week = ws.week AND ge.team = ws.team
        WHERE ws.season_type = 'REG' AND ws.position IN ('QB', 'RB', 'WR', 'TE')
    """)


def build_role_signal(column):
    return q(f"SELECT player_id, season, week, {column} AS signal_value FROM player_role_trend")


SIGNAL_BUILDERS = {
    "defense_vs_position.points_allowed_per_game_season_to_date": lambda lk: build_dvp_signal(lk),
    "game_environment.implied_margin": lambda lk: build_ge_signal("implied_margin"),
    "game_environment.wind": lambda lk: build_ge_signal("wind"),
    "game_environment.implied_team_total": lambda lk: build_ge_signal("implied_team_total"),
    "game_environment.temp": lambda lk: build_ge_signal("temp"),
    "player_role_trend.target_share_delta": lambda lk: build_role_signal("target_share_delta"),
    "player_role_trend.snap_share_delta": lambda lk: build_role_signal("snap_share_delta"),
}

comparison = []
for _, league in LEAGUES.iterrows():
    league_key = league["league_key"]
    actuals = build_actuals(league)
    projection = build_projection(league_key, actuals)
    for signal_name, builder in SIGNAL_BUILDERS.items():
        signal = builder(league_key)[["player_id", "season", "week", "signal_value"]]
        scored = score_signal(signal, actuals, projection)
        scored["signal"] = signal_name
        scored["league_key"] = league_key
        comparison.append(scored)

comparison = pd.concat(comparison, ignore_index=True)
comparison.shape

(186, 13)

In [5]:
proxy = comparison[
    (comparison["baseline"].isin(["season_to_date_ppg", "last3_ppg"]))
    & (comparison["league_key"] == "sleeper")
    & (comparison["position"] != "ALL")
][["signal", "position", "baseline", "n", "n_weeks", "incremental_rho", "t_stat", "p_value"]]
proxy.sort_values(["signal", "position", "baseline"]).reset_index(drop=True)

,signal,position,baseline,n,n_weeks,incremental_rho,t_stat,p_value
0,defense_vs_position.points_allowed_per_game_se...,QB,last3_ppg,4631,159,0.097707,6.242923,3.792650e-09
1,defense_vs_position.points_allowed_per_game_se...,QB,season_to_date_ppg,6035,181,0.087805,7.235146,1.293010e-11
2,defense_vs_position.points_allowed_per_game_se...,RB,last3_ppg,11723,159,0.066675,7.007319,6.628153e-11
3,defense_vs_position.points_allowed_per_game_se...,RB,season_to_date_ppg,14710,181,0.064245,6.916367,7.808446e-11
4,defense_vs_position.points_allowed_per_game_se...,TE,last3_ppg,9004,159,0.050066,4.469831,1.486611e-05
5,defense_vs_position.points_allowed_per_game_se...,TE,season_to_date_ppg,11497,181,0.040316,4.022391,8.468877e-05
6,defense_vs_position.points_allowed_per_game_se...,WR,last3_ppg,18723,159,0.032673,4.137878,5.683217e-05
7,defense_vs_position.points_allowed_per_game_se...,WR,season_to_date_ppg,23283,181,0.029802,4.144885,5.227631e-05
8,game_environment.implied_margin,QB,last3_ppg,4532,159,0.012661,0.899762,3.696162e-01
9,game_environment.implied_margin,QB,season_to_date_ppg,5913,181,-0.007188,-0.595813,5.520486e-01


Every number here matches what each module's own docstring already reports (to rounding): `defense_vs_position` clears `|t| >= 2` at every position against `season_to_date_ppg` (QB/RB/TE/WR incremental rho 0.088/0.064/0.040/0.030, all p < 0.0005); `game_environment`'s `implied_margin` clears it for RB and TE and `wind` clears it for QB and WR, while `implied_team_total` and `temp` don't survive the swap from `season_to_date_ppg` to `last3_ppg`; `player_role_trend`'s deltas flip sign between the two baselines at every position, the instability #134 already found. None of that is news — it's the same eleven-season, cross-league-confirmed finding each module already settled, reproduced fresh rather than quoted. What's new is putting all three on one table and, next, checking all three against the bar that actually matters.

## Applying the promotion gate: the `sleeper_points` baseline

This is where the three signals actually get judged, and none of them can be today — for two different, mechanical reasons, not because any one signal is weak.

`defense_vs_position` and `player_role_trend`'s `_delta` columns are **walk-forward**: both `defense_vs_position._walk_forward` and `player_role_trend._walk_forward` shift a season's series by one game before averaging, so the very first game of a season — the only 2026 game `weekly_stats` holds as of this run — has no prior game to compute a figure from. The signal itself is null there, structurally, independent of whether a vendor projection exists to hold fixed.

`game_environment`'s columns and `player_role_trend`'s **level** columns are not walk-forward — they describe the game or the player's role *as of* that week, not a trend into it — so they do have real values in 2026's one played week. But `score_signal` clusters its significance test by `(season, week)`, and one clustered week can't produce a t-statistic (`_score_group` returns `NaN` rather than a false read when fewer than two weeks have a value — see `test_weekly_backtest.py`'s own `test_significance_clusters_by_week_and_degrades_to_nan_with_one_week`). Real rows, still no number to gate on.

In [6]:
GATE_SIGNALS = {
    "defense_vs_position.points_allowed_per_game_season_to_date (walk-forward)": build_dvp_signal,
    "game_environment.implied_margin (same-week)": lambda lk: build_ge_signal("implied_margin"),
    "player_role_trend.target_share (level, same-week)": lambda lk: build_role_signal("target_share"),
    "player_role_trend.target_share_delta (walk-forward)": lambda lk: build_role_signal("target_share_delta"),
}

gate_rows = []
for _, league in LEAGUES.iterrows():
    league_key = league["league_key"]
    actuals = build_actuals(league)
    projection = build_projection(league_key, actuals)
    for signal_name, builder in GATE_SIGNALS.items():
        signal = builder(league_key)[["player_id", "season", "week", "signal_value"]]
        scored = score_signal(signal, actuals, projection)
        # ALL kept alongside every position: for the walk-forward signals every per-position row
        # vanishes at the dropna above (no signal value survives), so ALL is the only row left to
        # show the n=0 structural case at all rather than that signal silently having no rows here.
        scored = scored[scored["baseline"] == "sleeper_points"]
        scored["signal"] = signal_name
        scored["league_key"] = league_key
        gate_rows.append(scored)

gate_rows = pd.concat(gate_rows, ignore_index=True)
gate_rows["passes_gate"] = gate_rows.apply(passes_gate, axis=1)
gate_rows[["signal", "league_key", "position", "n", "n_weeks", "t_stat", "passes_gate"]]\
    .sort_values(["signal", "position", "league_key"]).reset_index(drop=True)

,signal,league_key,position,n,n_weeks,t_stat,passes_gate
0,defense_vs_position.points_allowed_per_game_se...,espn,ALL,0,0,NaN,False
1,defense_vs_position.points_allowed_per_game_se...,sleeper,ALL,0,0,NaN,False
2,game_environment.implied_margin (same-week),espn,ALL,326,1,NaN,False
3,game_environment.implied_margin (same-week),sleeper,ALL,326,1,NaN,False
4,game_environment.implied_margin (same-week),espn,QB,32,1,NaN,False
5,game_environment.implied_margin (same-week),sleeper,QB,32,1,NaN,False
6,game_environment.implied_margin (same-week),espn,RB,81,1,NaN,False
7,game_environment.implied_margin (same-week),sleeper,RB,81,1,NaN,False
8,game_environment.implied_margin (same-week),espn,TE,73,1,NaN,False
9,game_environment.implied_margin (same-week),sleeper,TE,73,1,NaN,False


## Verdict: no signal is promoted — a documented non-event, not a rejection

`passes_gate` is `False` on every row above, and mechanically so: the walk-forward signals show `n = 0` (no row has a non-null signal value yet), the same-week signals show real `n` but `n_weeks = 1`, short of the stated floor of 17 either way. Applying the identical rule to all three signals produces the identical answer for all three, for the identical shared reason — the warehouse doesn't yet hold enough of the 2026 season to run this specific test, not a finding about any one signal's quality.

This reconciles cleanly against what each module's own docstring already concludes, and none of them need to change:

- `defense_vs_position.py`'s verdict already says the effect "stays *display*, not *weighted* ...   because the actual promotion bar this epic sets ... is currently unanswerable" — the unified gate   agrees.
- `game_environment.py`'s verdict says the same, for the same reason, about `implied_margin` and   `wind`.
- `player_role_trend.py`'s verdict goes further and says no rule is supportable at all for its   direction signal, independent of the promotion bar — the unified gate doesn't disagree, it just   can't even be evaluated on that table yet either.

One correction, not a verdict change: each of the three docstrings says the warehouse holds *zero* player-weeks with both a completed game and a `sleeper_points` row. That's now slightly stale — it's one week (2026 week 1), not zero, as the `n > 0` rows above show for the same-week signals. The practical answer is unchanged (one clustered week can't produce a significance test either way), so nothing in those docstrings needs rewriting; the accurate, current count belongs here, in a notebook that recomputes it, rather than as a number frozen into a docstring the next warehouse rebuild would make wrong again.

## Provisional verdicts

The verdict against the epic's actual promotion bar — beating the vendor's own weekly projection — is **provisional for all three signals**, not settled, and for two different reasons:

- `defense_vs_position` and `player_role_trend`'s delta columns have never been tested against it at   all: their walk-forward construction means they carry no value yet in the only 2026 week played.
- `game_environment` and `player_role_trend`'s level columns have been tested on exactly one   clustered week — real rows, but short of even the minimum two weeks `score_signal` needs to   attempt a t-statistic, let alone this notebook's stated floor of 17.

**What unblocks it**: the 2026 in-season archive continuing to accumulate, one week at a time, now that #117 (closed) fixed the archiving-instead-of-overwriting bug that left `weekly_stats` holding no 2026 rows at all as recently as the three individual measurement tickets. No further ticket owns this gap — #117 already shipped the fix, and what's missing now is purely the season playing out. Re-run this notebook once `n_weeks >= 17` against the `sleeper_points` baseline for a walk-forward signal (around week 17 of a season that has actually reached that point in `weekly_stats`) or a same-week signal (as soon as two or more weeks have both actuals and a vendor projection).

The verdicts against the two proxy baselines (`season_to_date_ppg`, `last3_ppg`) are **not** provisional — they're the same eleven-season, cross-league-confirmed findings each module's own docstring already settled, reproduced fresh above, and nothing here changes them.

## What this means for `weekly_player_context`

Nothing changes there. Every `dvp_*`/`game_*`/`role_*` column it already carries stays exactly what `weekly_player_context.py`'s own docstring says it is: flat, unblended, displayed context, with "nothing here weighted or scored" by design — #114's whole point was to let promotion happen as a change to a future consumer, not a rebuild of that table.

Building a weighted consumer for `defense_vs_position` — the one signal that clears a `|t| >= 2` reliability bar robustly, at every position, against both available proxy baselines — would still be premature: that's not the bar this epic set, and reaching for the proxy result because the literal one isn't answerable yet is exactly the "promote whichever measured best" failure this ticket's own problem statement exists to refuse. If `defense_vs_position` (or anything else) clears the `sleeper_points` gate once the 2026 archive is deep enough, *that's* the finding worth proposing a new epic over — per #114's own Out of Scope, not this ticket's.